In [3]:
!pip install SPARQLWrapper rdflib networkx tqdm

In [4]:
import os
import re
import json
import time
import hashlib
import sqlite3
from typing import List, Set, Dict, Tuple
from urllib.parse import unquote
from SPARQLWrapper import SPARQLWrapper, JSON, POST
import rdflib
from rdflib import Graph, URIRef, Literal, Namespace
from rdflib.namespace import RDFS, RDF, SKOS, DCTERMS
import networkx as nx
from tqdm import tqdm


In [5]:
!rm -f dbpedia_extraction_cache.sqlite


In [6]:

# =====================================================================
# 1. CONFIGURAÇÃO E SEMENTES DE DOMÍNIO (DONES + AI PAPERS + MATH)
# =====================================================================

SPARQL_ENDPOINT = "https://dbpedia.org/sparql"
USER_AGENT = "IEEE-CIS-KG-RAG-Extractor/2.0 (https://cis.taskforce.ieee.org/)"
MAX_KG_SIZE_MB = 95.0
MAX_COMMENT_CHARS = 450
BATCH_SIZE = 25          # Lote padrão para metadados/propriedades
WIKILINK_BATCH_SIZE = 12 # Lote menor para wikiPageWikiLink (evita Virtuoso S1TAT)
CACHE_DB = "dbpedia_extraction_cache.sqlite"

DBO = Namespace("http://dbpedia.org/ontology/")
DBP = Namespace("http://dbpedia.org/property/")
DBR = Namespace("http://dbpedia.org/resource/")
DBC = Namespace("http://dbpedia.org/resource/Category:")

# Palavras-chave para bloquear subcategorias irrelevantes (Ficção, Biografias, Listas)
EXCLUDED_CAT_KEYWORDS = (
    "fictional", "comics", "anime", "manga", "films", "novels", "television",
    "video_games", "people", "scientists", "physicists", "mathematicians",
    "engineers", "researchers", "academics", "writers", "stubs", "lists",
    "organizations", "companies", "institutions", "universities", "awards",
    "conferences", "journals", "magazines", "books", "albums", "songs",
    "accidents", "disasters", "weapons", "military", "history_of"
)

# Prefixos de recursos que devem ser ignorados em Python (sem pesar no SPARQL)
INVALID_RESOURCE_PREFIXES = (
    "http://dbpedia.org/resource/Category:",
    "http://dbpedia.org/resource/File:",
    "http://dbpedia.org/resource/Template:",
    "http://dbpedia.org/resource/List_of_",
    "http://dbpedia.org/resource/Outline_of_",
    "http://dbpedia.org/resource/Index_of_",
    "http://dbpedia.org/resource/Portal:",
    "http://dbpedia.org/resource/Wikipedia:",
)

SEED_CATEGORIES = [
    # Pilar 1: IFMIF-DONES, Fusão Nuclear, Aceleradores, Neutrônica e Isótopos
    "dbc:Nuclear_fusion",
    "dbc:Particle_accelerators",
    "dbc:Accelerator_physics",
    "dbc:Neutron_sources",
    "dbc:Nuclear_physics",
    "dbc:Radioactivity",
    "dbc:Radiation_protection",
    "dbc:Nuclear_materials",
    "dbc:Isotopes_of_beryllium",
    "dbc:Isotopes_of_lithium",
    "dbc:Isotopes_of_hydrogen",
    "dbc:Heat_exchangers",
    "dbc:Heat_transfer",
    "dbc:Fluid_dynamics",
    "dbc:Scintillators",
    "dbc:Particle_detectors",
    # Pilar 2: IA, Inteligência Computacional, Deep Learning e XAI
    "dbc:Machine_learning",
    "dbc:Deep_learning",
    "dbc:Artificial_neural_networks",
    "dbc:Computational_intelligence",
    "dbc:Fuzzy_logic",
    "dbc:Evolutionary_computation",
    "dbc:Explainable_artificial_intelligence",
    "dbc:Federated_learning",
    "dbc:Statistical_classification",
    "dbc:Cluster_analysis",
    "dbc:Dimension_reduction",
    "dbc:Reinforcement_learning",
    "dbc:Natural_language_processing",
    "dbc:Generative_artificial_intelligence",
    "dbc:Markov_models",
    "dbc:Control_theory",
    # Pilar 3: Análise Funcional, Espaços de Banach e Estatística Bayesiana
    "dbc:Banach_spaces",
    "dbc:Functional_analysis",
    "dbc:Normed_spaces",
    "dbc:Operator_theory",
    "dbc:Metric_geometry",
    "dbc:Convex_analysis",
    "dbc:Bayesian_statistics",
    "dbc:Monte_Carlo_methods",
    "dbc:Mathematical_optimization",
]

SEED_ENTITIES = [
    # DONES & Física
    "dbr:International_Fusion_Materials_Irradiation_Facility",
    "dbr:DEMO",
    "dbr:ITER",
    "dbr:Transition_radiation",
    "dbr:Scintillation_(physics)",
    "dbr:Scintillator",
    "dbr:Luminescence",
    "dbr:Dielectric",
    "dbr:Beryllium-7",
    "dbr:Tritium",
    "dbr:Deuterium",
    "dbr:Lithium",
    "dbr:Lithium-6",
    "dbr:Lithium-7",
    "dbr:Half-life",
    "dbr:Exponential_decay",
    "dbr:Radioactive_decay",
    "dbr:Neutron_activation",
    "dbr:Neutron_flux",
    "dbr:Cross_section_(physics)",
    "dbr:Heat_exchanger",
    "dbr:Volumetric_flow_rate",
    "dbr:Mass_flow_rate",
    "dbr:Cold_trap",
    "dbr:Sievert",
    "dbr:Becquerel",
    "dbr:Absorbed_dose",
    "dbr:Equivalent_dose",
    "dbr:Remote_manipulator",
    "dbr:Monte_Carlo_N-Particle_Transport_Code",
    # IA & Inteligência Computacional
    "dbr:Fuzzy_cognitive_map",
    "dbr:Cognitive_map",
    "dbr:Sigmoid_function",
    "dbr:Softmax_function",
    "dbr:Hinge_loss",
    "dbr:Counterfactual_conditional",
    "dbr:Explainable_artificial_intelligence",
    "dbr:Variational_autoencoder",
    "dbr:Vector_quantization",
    "dbr:K-means_clustering",
    "dbr:Self-supervised_learning",
    "dbr:Diffusion_model",
    "dbr:Zero-shot_learning",
    "dbr:Transformer_(deep_learning_architecture)",
    "dbr:State-space_representation",
    "dbr:Linear_time-invariant_system",
    "dbr:Convolution",
    "dbr:Byzantine_fault",
    "dbr:Federated_learning",
    "dbr:Stochastic_gradient_descent",
    "dbr:Geometric_median",
    "dbr:Active_learning_(machine_learning)",
    "dbr:Artificial_general_intelligence",
    "dbr:Fairness_(machine_learning)",
    # Matemática & Estatística
    "dbr:Banach_space",
    "dbr:Normed_vector_space",
    "dbr:Dual_space",
    "dbr:Unit_sphere",
    "dbr:Unit_ball",
    "dbr:Dense_set",
    "dbr:Numerical_range",
    "dbr:Bayesian_inference",
    "dbr:Credible_interval",
    "dbr:Bernoulli_distribution",
    "dbr:Region_of_practical_equivalence",
]

TARGET_PROPERTIES = [
    "dbo:halfLife",
    "dbp:halfLife",
    "dbo:density",
    "dbp:density",
    "dbp:atomicMass",
    "dbp:decayMode",
    "dbp:meltingPoint",
    "dbp:boilingPoint",
    "dbp:symbol",
    "dbo:symbol",
    "dbo:meaning",
    "dbo:academicDiscipline",
    "dbo:field",
    "dbo:knownFor",
    "dbo:algorithm",
    "dbo:type",
]


# =====================================================================
# 2. CLIENTE SPARQL TOLERANTE A FALHAS (COM RECUPERAÇÃO DE S1TAT)
# =====================================================================

class DBpediaExtractor:
    def __init__(self, endpoint: str = SPARQL_ENDPOINT, cache_db: str = CACHE_DB):
        self.sparql = SPARQLWrapper(endpoint, agent=USER_AGENT)
        self.sparql.setReturnFormat(JSON)
        self.sparql.setMethod(POST)  # Evita erro de URI longa
        self.sparql.setTimeout(45)
        # Parâmetro nativo do Virtuoso para limitar tempo de execução sem derrubar HTTP
        self.sparql.addParameter("timeout", "25000")
        self.conn = sqlite3.connect(cache_db)
        self._init_cache()

    def _init_cache(self):
        with self.conn:
            self.conn.execute(
                "CREATE TABLE IF NOT EXISTS query_cache (query_hash TEXT PRIMARY KEY, response TEXT)"
            )

    @staticmethod
    def _hash_query(q: str) -> str:
        """Hash determinístico entre execuções usando SHA-256."""
        return hashlib.sha256(q.encode("utf-8")).hexdigest()

    @staticmethod
    def _salvage_partial_virtuoso_json(raw_bytes: bytes) -> List[Dict]:
        """
        Se o Virtuoso interromper a resposta no meio com 'Virtuoso S1TAT Error',
        recupera todas as linhas JSON completas enviadas antes do corte.
        """
        text = raw_bytes.decode("utf-8", errors="ignore")
        if '"bindings"' not in text:
            return []
        bindings_Part = text.split('"bindings"')[1]
        recovered = []
        for line in bindings_Part.splitlines():
            line = line.strip().rstrip(",")
            if line.startswith("{") and line.endswith("}"):
                try:
                    obj = json.loads(line)
                    if isinstance(obj, dict):
                        recovered.append(obj)
                except Exception:
                    continue
        return recovered

    def query(self, sparql_query: str, retries: int = 3) -> List[Dict]:
        q_hash = self._hash_query(sparql_query)
        cur = self.conn.cursor()
        cur.execute("SELECT response FROM query_cache WHERE query_hash = ?", (q_hash,))
        row = cur.fetchone()
        if row is not None:
            return json.loads(row[0])

        for attempt in range(retries):
            try:
                self.sparql.setQuery(sparql_query)
                raw_response = self.sparql.query()
                results = raw_response.convert()
                bindings = results.get("results", {}).get("bindings", [])
                with self.conn:
                    self.conn.execute(
                        "INSERT OR REPLACE INTO query_cache VALUES (?, ?)",
                        (q_hash, json.dumps(bindings)),
                    )
                time.sleep(0.15)
                return bindings
            except Exception as e:
                err_str = str(e)
                # Verifica se é um erro S1TAT com JSON parcial na mensagem da exceção
                if "S1TAT" in err_str or "EndPointInternalError" in err_str:
                    raw_bytes = getattr(e, "response", b"")
                    if not raw_bytes and "Response:\nb'" in err_str:
                        raw_bytes = err_str.encode("utf-8", errors="ignore")
                    salvaged = self._salvage_partial_virtuoso_json(
                        raw_bytes if isinstance(raw_bytes, bytes) else str(raw_bytes).encode("utf-8")
                    )
                    if salvaged:
                        print(f"[Info] Virtuoso S1TAT timeout: recuperadas {len(salvaged)} tuplas parciais.")
                        with self.conn:
                            self.conn.execute(
                                "INSERT OR REPLACE INTO query_cache VALUES (?, ?)",
                                (q_hash, json.dumps(salvaged)),
                            )
                        return salvaged

                wait = 2 ** (attempt + 1)
                if attempt == retries - 1:
                    raise e
                time.sleep(wait)
        return []

    def query_batch_adaptive(self, uris: List[str], build_query_fn) -> List[Dict]:
        """
        Executa uma consulta SPARQL sobre um lote de URIs.
        Se o lote causar erro 500/timeout no Virtuoso, divide o lote ao meio recursivamente!
        """
        if not uris:
            return []
        q = build_query_fn(uris)
        try:
            return self.query(q, retries=2)
        except Exception as e:
            if len(uris) == 1:
                print(f"[Skip] URI muito pesada ignorada ({uris[0]}): {str(e)[:120]}")
                return []
            mid = len(uris) // 2
            print(f"[Split] Dividindo lote pesado de {len(uris)} -> {mid} + {len(uris) - mid} URIs...")
            left = self.query_batch_adaptive(uris[:mid], build_query_fn)
            right = self.query_batch_adaptive(uris[mid:], build_query_fn)
            return left + right

    # -----------------------------------------------------------------
    # PASSO 1: Expansão de Categorias SKOS (Com Filtro Anti-Ruído)
    # -----------------------------------------------------------------
    @staticmethod
    def _is_valid_category(cat_uri: str) -> bool:
        cat_low = cat_uri.lower()
        return not any(kw in cat_low for kw in EXCLUDED_CAT_KEYWORDS)

    @staticmethod
    def _is_valid_resource(ent_uri: str) -> bool:
        if any(ent_uri.startswith(p) for p in INVALID_RESOURCE_PREFIXES):
            return False
        ent_low = ent_uri.lower()
        if any(bad in ent_low for bad in ("_(comics)", "_(film)", "_(character)", "_(novel)", "_(band)")):
            return False
        return True

    def expand_categories(self, seed_cats: List[str], depth: int = 1) -> Tuple[Set[str], List[Tuple[str, str]]]:
        all_categories = {
            c.replace("dbc:", "http://dbpedia.org/resource/Category:") for c in seed_cats
        }
        frontier = set(all_categories)
        skos_edges: List[Tuple[str, str]] = []

        for d in range(depth):
            print(f"Expandindo categorias SKOS na profundidade {d+1} (fronteira: {len(frontier)})...")
            next_frontier = set()
            frontier_list = list(frontier)

            for i in tqdm(range(0, len(frontier_list), BATCH_SIZE), desc=f"SKOS Depth {d+1}"):
                batch = frontier_list[i : i + BATCH_SIZE]

                def make_q(b_uris):
                    v_str = " ".join(f"<{u}>" for u in b_uris)
                    return f"""
                    PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
                    SELECT ?subCat ?parentCat WHERE {{
                        VALUES ?parentCat {{ {v_str} }}
                        ?subCat skos:broader ?parentCat .
                    }}
                    LIMIT 8000
                    """

                rows = self.query_batch_adaptive(batch, make_q)
                for r in rows:
                    sub = r["subCat"]["value"]
                    parent = r["parentCat"]["value"]
                    if not self._is_valid_category(sub):
                        continue
                    skos_edges.append((sub, parent))
                    if sub not in all_categories:
                        next_frontier.add(sub)
                        all_categories.add(sub)
            frontier = next_frontier
            if not frontier:
                break

        print(f"Total de categorias técnicas filtradas: {len(all_categories)}")
        return all_categories, list(set(skos_edges))

    # -----------------------------------------------------------------
    # PASSO 2: Coleta de Entidades (Excluindo Pessoas, Organizações e Ficção)
    # -----------------------------------------------------------------
    def fetch_entities_from_categories(
        self, categories: Set[str]
    ) -> Tuple[Set[str], List[Tuple[str, str]]]:
        entities: Set[str] = {
            e.replace("dbr:", "http://dbpedia.org/resource/") for e in SEED_ENTITIES
        }
        subject_edges: List[Tuple[str, str]] = []
        cat_list = list(categories)

        for i in tqdm(range(0, len(cat_list), BATCH_SIZE), desc="Buscando entidades das categorias"):
            batch = cat_list[i : i + BATCH_SIZE]

            def make_q(b_uris):
                v_str = " ".join(f"<{u}>" for u in b_uris)
                # Sem FILTER(STRSTARTS) pesado! Apenas exclui biografias (dbo:Person) e ficção
                return f"""
                PREFIX dct: <http://purl.org/dc/terms/>
                PREFIX dbo: <http://dbpedia.org/ontology/>
                PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
                SELECT ?entity ?cat WHERE {{
                    VALUES ?cat {{ {v_str} }}
                    ?entity dct:subject ?cat .
                    FILTER NOT EXISTS {{ ?entity rdf:type dbo:Person }}
                    FILTER NOT EXISTS {{ ?entity rdf:type dbo:FictionalCharacter }}
                    FILTER NOT EXISTS {{ ?entity rdf:type dbo:Organisation }}
                }}
                LIMIT 6000
                """

            rows = self.query_batch_adaptive(batch, make_q)
            for r in rows:
                ent = r["entity"]["value"]
                cat = r["cat"]["value"]
                if self._is_valid_resource(ent):
                    entities.add(ent)
                    subject_edges.append((ent, cat))

        print(f"Total de entidades candidatas (limpas): {len(entities)}")
        return entities, list(set(subject_edges))

    # -----------------------------------------------------------------
    # PASSO 3: Labels, Definições Curtas, Propriedades e Redirecionamentos
    # -----------------------------------------------------------------
    def fetch_entity_metadata_and_properties(
        self, entities: Set[str]
    ) -> Tuple[Dict[str, Dict[str, str]], List[Tuple[str, str, str, bool]], Dict[str, List[str]]]:
        metadata: Dict[str, Dict[str, str]] = {}
        prop_triples: List[Tuple[str, str, str, bool]] = []
        aliases_by_ent: Dict[str, List[str]] = {}
        ent_list = list(entities)

        prop_uris = " ".join(
            f"<{p.replace('dbo:', str(DBO)).replace('dbp:', str(DBP))}>"
            for p in TARGET_PROPERTIES
        )

        for i in tqdm(range(0, len(ent_list), BATCH_SIZE), desc="Extraindo labels, definições e constantes"):
            batch = ent_list[i : i + BATCH_SIZE]

            # 1. Labels e Comments em Inglês
            def make_meta_q(b_uris):
                v_str = " ".join(f"<{u}>" for u in b_uris)
                return f"""
                PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
                SELECT ?entity ?label ?comment WHERE {{
                    VALUES ?entity {{ {v_str} }}
                    ?entity rdfs:label ?label .
                    FILTER(LANG(?label) = "en")
                    OPTIONAL {{
                        ?entity rdfs:comment ?comment .
                        FILTER(LANG(?comment) = "en")
                    }}
                }}
                """

            for r in self.query_batch_adaptive(batch, make_meta_q):
                ent = r["entity"]["value"]
                lbl = r["label"]["value"]
                cmt = r.get("comment", {}).get("value", "")
                if len(cmt) > MAX_COMMENT_CHARS:
                    cmt = cmt[:MAX_COMMENT_CHARS].rsplit(" ", 1)[0] + "..."
                metadata[ent] = {"label": lbl, "comment": cmt}

            # 2. Propriedades Físicas, Químicas e Matemáticas
            def make_prop_q(b_uris):
                v_str = " ".join(f"<{u}>" for u in b_uris)
                return f"""
                SELECT ?entity ?prop ?val WHERE {{
                    VALUES ?entity {{ {v_str} }}
                    VALUES ?prop {{ {prop_uris} }}
                    ?entity ?prop ?val .
                    FILTER(!isLiteral(?val) || LANG(?val) = "" || LANG(?val) = "en")
                }}
                """

            for r in self.query_batch_adaptive(batch, make_prop_q):
                ent = r["entity"]["value"]
                prop = r["prop"]["value"]
                val_obj = r["val"]
                is_uri = val_obj["type"] == "uri"
                prop_triples.append((ent, prop, val_obj["value"], is_uri))

            # 3. Siglas e Redirecionamentos Curtos (ex: OTR -> Transition_radiation, Be-7 -> Beryllium-7)
            def make_redir_q(b_uris):
                v_str = " ".join(f"<{u}>" for u in b_uris)
                return f"""
                PREFIX dbo: <http://dbpedia.org/ontology/>
                SELECT ?redirect ?entity WHERE {{
                    VALUES ?entity {{ {v_str} }}
                    ?redirect dbo:wikiPageRedirects ?entity .
                }}
                LIMIT 5000
                """

            for r in self.query_batch_adaptive(batch, make_redir_q):
                ent = r["entity"]["value"]
                redir_uri = r["redirect"]["value"]
                alias_str = unquote(redir_uri.split("/")[-1].replace("_", " "))
                if 2 <= len(alias_str) <= 35:
                    aliases_by_ent.setdefault(ent, []).append(alias_str)

        return metadata, prop_triples, aliases_by_ent

    # -----------------------------------------------------------------
    # PASSO 4: Subgrafo Induzido de wikiPageWikiLink (À Prova de Timeout)
    # -----------------------------------------------------------------
    def fetch_induced_wikilinks(self, valid_entities: Set[str]) -> List[Tuple[str, str]]:
        """
        Consulta sem FILTER(STRSTARTS) e sem DISTINCT no SPARQL (evita S1TAT Error).
        A filtragem de domínio (`dst in valid_entities`) e deduplicação ocorrem em Python em O(1).
        """
        induced_edges: Set[Tuple[str, str]] = set()
        ent_list = list(valid_entities)

        for i in tqdm(range(0, len(ent_list), WIKILINK_BATCH_SIZE), desc="Extraindo wikiPageWikiLinks induzidos"):
            batch = ent_list[i : i + WIKILINK_BATCH_SIZE]

            def make_link_q(b_uris):
                v_str = " ".join(f"<{u}>" for u in b_uris)
                return f"""
                PREFIX dbo: <http://dbpedia.org/ontology/>
                SELECT ?src ?dst WHERE {{
                    VALUES ?src {{ {v_str} }}
                    ?src dbo:wikiPageWikiLink ?dst .
                }}
                LIMIT 9500
                """

            rows = self.query_batch_adaptive(batch, make_link_q)
            for r in rows:
                src = r.get("src", {}).get("value")
                dst = r.get("dst", {}).get("value")
                if src and dst and src != dst and dst in valid_entities:
                    induced_edges.add((src, dst))

        print(f"Total de arestas internas induzidas extraídas: {len(induced_edges)}")
        return list(induced_edges)


# =====================================================================
# 3. PODA POR CENTRALIDADE (PAGERANK) E SERIALIZAÇÃO (< 95 MB)
# =====================================================================

def build_and_prune_kg(
    metadata: Dict[str, Dict[str, str]],
    skos_edges: List[Tuple[str, str]],
    subject_edges: List[Tuple[str, str]],
    prop_triples: List[Tuple[str, str, str, bool]],
    induced_edges: List[Tuple[str, str]],
    aliases_by_ent: Dict[str, List[str]],
    output_ttl: str = "domain_kg.ttl",
    output_jsonl: str = "kg_rag_chunks.jsonl",
):
    nx_graph = nx.DiGraph()
    for src, dst in induced_edges:
        nx_graph.add_edge(src, dst)
    for ent, cat in subject_edges:
        if ent in metadata:
            nx_graph.add_edge(ent, cat)

    print("Calculando PageRank no grafo de domínio...")
    pagerank = nx.pagerank(nx_graph, alpha=0.85) if len(nx_graph) > 0 else {}

    seed_uris = {e.replace("dbr:", str(DBR)) for e in SEED_ENTITIES}
    for s in seed_uris:
        pagerank[s] = pagerank.get(s, 0.0) + 10.0

    ranked_entities = sorted(
        metadata.keys(), key=lambda u: pagerank.get(u, 0.0), reverse=True
    )

    keep_ratio = 1.0
    while keep_ratio > 0.1:
        cutoff = int(len(ranked_entities) * keep_ratio)
        active_entities = set(ranked_entities[:cutoff]) | seed_uris

        g = Graph()
        g.bind("dbr", DBR)
        g.bind("dbo", DBO)
        g.bind("dbp", DBP)
        g.bind("dbc", DBC)
        g.bind("skos", SKOS)
        g.bind("dct", DCTERMS)
        g.bind("rdfs", RDFS)

        for sub, parent in skos_edges:
            g.add((URIRef(sub), SKOS.broader, URIRef(parent)))
            cat_label = unquote(sub.split("Category:")[-1].replace("_", " "))
            g.add((URIRef(sub), RDFS.label, Literal(cat_label, lang="en")))

        for ent in active_entities:
            if ent not in metadata:
                continue
            u_ent = URIRef(ent)
            g.add((u_ent, RDFS.label, Literal(metadata[ent]["label"], lang="en")))
            if metadata[ent]["comment"]:
                g.add((u_ent, RDFS.comment, Literal(metadata[ent]["comment"], lang="en")))

        for ent, cat in subject_edges:
            if ent in active_entities:
                g.add((URIRef(ent), DCTERMS.subject, URIRef(cat)))

        for ent, prop, val, is_uri in prop_triples:
            if ent in active_entities:
                obj = URIRef(val) if is_uri else Literal(val)
                g.add((URIRef(ent), URIRef(prop), obj))

        for src, dst in induced_edges:
            if src in active_entities and dst in active_entities:
                g.add((URIRef(src), DBO.wikiPageWikiLink, URIRef(dst)))

        print(f"Serializando grafo RDF ({len(g)} triplas) em {output_ttl}...")
        g.serialize(destination=output_ttl, format="turtle")

        size_mb = os.path.getsize(output_ttl) / (1024 * 1024)
        print(f"Tamanho atual do KG: {size_mb:.2f} MB (Limite: {MAX_KG_SIZE_MB:.2f} MB)")

        if size_mb <= MAX_KG_SIZE_MB:
            print(f"Sucesso! KG dentro do orçamento ({size_mb:.2f} MB).")
            export_rag_jsonl(
                active_entities, metadata, prop_triples, induced_edges,
                subject_edges, aliases_by_ent, output_jsonl
            )
            break
        else:
            print("Acima de 95 MB; podando 15% dos nós de menor centralidade...")
            keep_ratio *= 0.85


def export_rag_jsonl(
    active_entities: Set[str],
    metadata: Dict[str, Dict[str, str]],
    prop_triples: List[Tuple[str, str, str, bool]],
    induced_edges: List[Tuple[str, str]],
    subject_edges: List[Tuple[str, str]],
    aliases_by_ent: Dict[str, List[str]],
    output_jsonl: str,
):
    props_by_ent: Dict[str, List[str]] = {}
    for ent, prop, val, _ in prop_triples:
        if ent in active_entities:
            p_name = prop.split("/")[-1]
            v_clean = unquote(val.split("/")[-1].replace("_", " ")) if val.startswith("http") else val
            props_by_ent.setdefault(ent, []).append(f"{p_name}: {v_clean}")

    neighbors_by_ent: Dict[str, List[str]] = {}
    for src, dst in induced_edges:
        if src in active_entities and dst in active_entities:
            dst_label = metadata.get(dst, {}).get("label", unquote(dst.split("/")[-1].replace("_", " ")))
            neighbors_by_ent.setdefault(src, []).append(dst_label)

    cats_by_ent: Dict[str, List[str]] = {}
    for ent, cat in subject_edges:
        if ent in active_entities:
            c_name = unquote(cat.split("Category:")[-1].replace("_", " "))
            cats_by_ent.setdefault(ent, []).append(c_name)

    with open(output_jsonl, "w", encoding="utf-8") as f:
        for ent in active_entities:
            if ent not in metadata:
                continue
            record = {
                "uri": ent,
                "label": metadata[ent]["label"],
                "aliases": list(set(aliases_by_ent.get(ent, [])))[:15],
                "definition": metadata[ent]["comment"],
                "categories": list(set(cats_by_ent.get(ent, [])))[:10],
                "properties": list(set(props_by_ent.get(ent, [])))[:15],
                "related_concepts": list(set(neighbors_by_ent.get(ent, [])))[:25],
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
    print(f"Exportados cartões KG-RAG (com aliases) em {output_jsonl}")


if __name__ == "__main__":
    extractor = DBpediaExtractor()

    categories, skos_edges = extractor.expand_categories(SEED_CATEGORIES, depth=1)
    entities, subject_edges = extractor.fetch_entities_from_categories(categories)
    metadata, prop_triples, aliases_by_ent = extractor.fetch_entity_metadata_and_properties(entities)

    valid_entities = set(metadata.keys())
    induced_edges = extractor.fetch_induced_wikilinks(valid_entities)

    build_and_prune_kg(
        metadata=metadata,
        skos_edges=skos_edges,
        subject_edges=subject_edges,
        prop_triples=prop_triples,
        induced_edges=induced_edges,
        aliases_by_ent=aliases_by_ent,
    )

Expandindo categorias SKOS na profundidade 1 (fronteira: 41)...


SKOS Depth 1: 100%|██████████████████████████████████████████████████████████████████████| 2/2 [00:03<00:00,  1.84s/it]


Total de categorias técnicas filtradas: 261


Buscando entidades das categorias:  36%|█████████████████▍                              | 4/11 [01:40<03:43, 31.87s/it]

[Split] Dividindo lote pesado de 25 -> 12 + 13 URIs...


Buscando entidades das categorias: 100%|███████████████████████████████████████████████| 11/11 [02:55<00:00, 15.95s/it]


... (29 outputs left out)

Extraindo wikiPageWikiLinks induzidos:  97%|███████████████████████████████████████▊ | 675/696 [53:41<02:30,  7.15s/it]

[Split] Dividindo lote pesado de 12 -> 6 + 6 URIs...


Extraindo wikiPageWikiLinks induzidos: 100%|███████████████████████████████████████| 696/696 [1:13:11<00:00,  6.31s/it]


Total de arestas internas induzidas extraídas: 60021
Calculando PageRank no grafo de domínio...
Serializando grafo RDF (79235 triplas) em domain_kg.ttl...
Tamanho atual do KG: 2.93 MB (Limite: 95.00 MB)
Sucesso! KG dentro do orçamento (2.93 MB).
Exportados cartões KG-RAG (com aliases) em kg_rag_chunks.jsonl
